In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Window-Function-Demo'). \
    master('yarn'). \
    getOrCreate()

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Date-Time-Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [5]:
sc = spark.sparkContext

# pyspark.sql.functions.to_json

<img src="images/json_tuple.jpg" align = "left">



Ref - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.json_tuple.html

While `from_json` requires a schema, `json_tuple` is a more specialized generator function used to extract multiple fields from a JSON string in one go. Think of it as a "shortcut" for when you have a flat JSON structure and want to pull out specific keys without the overhead of defining a full `StructType`.

# 1. How it Works

The function takes a JSON column and a series of string arguments representing the keys you want to extract.

1. **Efficiency:** It is often faster than calling `get_json_object` multiple times because it parses the JSON string only once per row to retrieve all requested fields.

2. **Behavior:** It returns a Table Generating Function (UDTF). In simple terms, it produces multiple columns at once.

3. **Strictness:** It only works for top-level keys. It cannot traverse nested paths (like `user.address.city`).

# 2. Working Example

Let's extract specific fields from a raw JSON string without defining a schema.

**Step 1:** Create Data  


In [8]:
from pyspark.sql.functions import json_tuple, col

data = [
    (1, '{"name": "Alice", "age": 30, "city": "New York", "job": "Engineer"}'),
    (2, '{"name": "Bob", "age": 25, "city": "Seattle", "job": "Designer"}')
]
df = spark.createDataFrame(data, ["id", "raw_json"])

df.printSchema()
df.show()

root
 |-- id: long (nullable = true)
 |-- raw_json: string (nullable = true)

+---+--------------------+
| id|            raw_json|
+---+--------------------+
|  1|{"name": "Alice",...|
|  2|{"name": "Bob", "...|
+---+--------------------+



**Step 2:** Apply `json_tuple`

Note that because `json_tuple` is a UDTF, you typically use it inside a `select` statement.

In [9]:
# Extract name, age, and job in one call
# Note: You don't need to wrap the fields in col() for the keys
result_df = df.select(
    col("id"), 
    json_tuple(col("raw_json"), "name", "age", "job")
).toDF("id", "name", "age", "job") # Rename columns as json_tuple returns c0, c1, etc.

result_df.show()

+---+-----+---+--------+
| id| name|age|     job|
+---+-----+---+--------+
|  1|Alice| 30|Engineer|
|  2|  Bob| 25|Designer|
+---+-----+---+--------+



# 3. Comparison: `json_tuple` vs `from_json`

- **Structs are best:** While you can use to_json on Maps, using struct() allows you to mix different data types (strings, ints, booleans) into the same JSON object.

- **Options:** You can pass a map of options to customize the output, such as {"timestampFormat": "yyyy-MM-dd"} to control how dates look inside the string.

- **Kafka Integration:** If you are writing to Kafka, your DataFrame must have a column named value of type string or binary. `to_json` is the standard way to prepare your data for that value column.

<img src="images/summary_comparison2.jpg" align = "left">

# When should you use it?

Use json_tuple when:

- Your JSON is flat (no nested objects).

- You only need a few specific fields from a large JSON blob.

- You want to avoid the "boilerplate" code of writing a StructType schema.